# Congressional Trade Machine Learning Modeling
This notebook implements and evaluates machine learning models to predict whether congressional stock trades outperform the market (S&P 500).

We focus on Logistic Regression and Random Forest models to identify key features associated with high-performing trades.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay, ConfusionMatrixDisplay
)
import sys
import os

# Add src to path to import our modeling functions
sys.path.append(os.path.abspath('../src'))
from models import load_data, prepare_features, build_pipeline

%matplotlib inline
sns.set_theme(style="whitegrid")

print("success")

ModuleNotFoundError: No module named 'seaborn'

## 1. Load data
We load both the trade features dataset and calculated returns dataset.


In [ ]:
# Load data using the function from src/models.py
# Note: we need to specify the path relative to the notebook
df = load_data('../data/trades_with_returns.csv')

# Display target distributions
targets = ['outperformed_30d', 'outperformed_60d', 'outperformed_90d']
plt.figure(figsize=(15, 5))

for i, target in enumerate(targets):
    plt.subplot(1, 3, i+1)
    if target in df.columns:
        counts = df[target].value_counts(normalize=True) * 100
        sns.barplot(x=counts.index.map({0.0: 'Under/Market', 1.0: 'Outperformed'}), y=counts.values, hue=counts.index, palette='viridis', legend=False)
        plt.title(f'{target} Distribution')
        plt.ylabel('Percentage (%)')
        plt.ylim(0, 100)
    else:
        plt.text(0.5, 0.5, f'{target} not found', ha='center')

plt.tight_layout()
plt.show()


## 2. Logistic Regression Training
Train the Logistic Regression model using the pipeline defined in `src/models.py`. Start with `outperformed_90d` as primary target.


In [ ]:
target_col = 'outperformed_90d'
X, y, num_features, cat_features = prepare_features(df, target_col=target_col)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build and train the pipeline
pipeline = build_pipeline(num_features, cat_features)
pipeline.fit(X_train, y_train)

print(f"Model trained on {len(X_train)} samples to predict {target_col}")


## 3. Model Evaluation
Evaluate the model's performance on an unseen test set.


In [ ]:
# Predictions
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Display metrics
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# Visualizations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
ConfusionMatrixDisplay.from_estimator(pipeline, X_test, y_test, cmap='Blues', ax=ax1)
ax1.set_title('Confusion Matrix')

# ROC Curve
RocCurveDisplay.from_estimator(pipeline, X_test, y_test, ax=ax2)
ax2.plot([0, 1], [0, 1], linestyle='--', color='red', label='Random')
ax2.set_title('ROC Curve')
ax2.legend()

plt.tight_layout()
plt.show()


## 4. Feature Importance
Analyze which features most strongly influence the model's predictions. Positive coefficients indicate features associated with a higher probability of outperforming the market.


In [ ]:
# Extract coefficients from the trained model
classifier = pipeline.named_steps['classifier']
preprocessor = pipeline.named_steps['preprocessor']

# Get feature names after one-hot encoding
onehot_cols = (preprocessor
               .named_transformers_['cat']
               .named_steps['onehot']
               .get_feature_names_out(cat_features))

feature_names = num_features + list(onehot_cols)
coeffs = classifier.coef_[0]

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coeffs,
    'Abs_Coefficient': np.abs(coeffs)
}).sort_values(by='Coefficient', ascending=False)

# Plot top positive and negative coefficients
plt.figure(figsize=(12, 8))
top_features = pd.concat([importance_df.head(10), importance_df.tail(10)])
sns.barplot(data=top_features, x='Coefficient', y='Feature', hue='Coefficient', palette='coolwarm', legend=False)
plt.title('Top Feature Coefficients (Logistic Regression)')
plt.axvline(0, color='black', linestyle='-', linewidth=1)
plt.show()


## 5. Random Forest
Random Forest can capture interactions between features that Logistic Regression might miss.


In [ ]:
# Build and train Random Forest pipeline
rf_pipeline = build_pipeline(num_features, cat_features, model_type='random_forest')
rf_pipeline.fit(X_train, y_train)

# Evaluation
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print("--- Random Forest Classification Report ---")
print(classification_report(y_test, y_pred_rf))

# Visual Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve comparison
RocCurveDisplay.from_estimator(pipeline, X_test, y_test, ax=ax1, name='Logistic Regression')
RocCurveDisplay.from_estimator(rf_pipeline, X_test, y_test, ax=ax1, name='Random Forest')
ax1.plot([0, 1], [0, 1], linestyle='--', color='red')
ax1.set_title('ROC Curve Comparison')

# RF Feature Importance
rf_classifier = rf_pipeline.named_steps['classifier']
rf_importances = rf_classifier.feature_importances_
rf_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_importances
}).sort_values(by='Importance', ascending=False)

sns.barplot(data=rf_importance_df.head(15), x='Importance', y='Feature', hue='Importance', palette='viridis', ax=ax2, legend=False)
ax2.set_title('Top 15 Feature Importances (Random Forest)')

plt.tight_layout()
plt.show()


## 6. Comparison Across Periods
Compare feature prediction success at 30, 60, and 90 days


In [ ]:
horizons = ['outperformed_30d', 'outperformed_60d', 'outperformed_90d']
results = []

for target in horizons:
    if target not in df.columns:
        continue
        
    X_h, y_h, num_h, cat_h = prepare_features(df, target_col=target)
    
    # Simple split for comparison
    X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
        X_h, y_h, test_size=0.2, random_state=42, stratify=y_h
    )
    
    # Results for both models
    for model_type in ['logistic_regression', 'random_forest']:
        pipe_h = build_pipeline(num_h, cat_h, model_type=model_type)
        pipe_h.fit(X_train_h, y_train_h)
        
        y_prob_h = pipe_h.predict_proba(X_test_h)[:, 1]
        auc = roc_auc_score(y_test_h, y_prob_h)
        
        results.append({
            'Horizon': target,
            'Model': model_type,
            'AUC-ROC': auc,
            'Positive Class %': y_h.mean() * 100
        })

results_df = pd.DataFrame(results)
print("Model Performance Across Time Horizons:")
print(results_df.to_string(index=False))


## 6. Conclusion
- **Model Baseline**: The Logistic Regression provides a starting point for understanding congressional trade performance.
- **Key Indicators**: Leadership positions, specific committees, and trade size are key features to watch.
- **Class Imbalance**: Many horizons show an imbalance (only ~30% outperform)
- **Next Models**: Future work will explore other models
